# Python Memory Management: Allocation, Deallocation and Garbage Collection

## Learning objectives

By the end of this notebook, you should be able to:

- Explain how Python objects are created, stored, referenced and reclaimed.
- Distinguish a variable name from the object it refers to.
- Understand reference counting and cyclic garbage collection.
- Use `id()`, `sys.getrefcount()`, `gc`, `weakref` and `tracemalloc` for investigation.
- Identify memory leaks, object retention and common causes of unnecessary memory usage.
- Apply memory-efficient techniques to data processing and ML workflows.

> **Implementation note:** The examples describe CPython where implementation-specific behavior is involved. Python's language-level guarantees are broader; other Python implementations may manage memory differently.


## 1. What is memory management?

Memory management is the process of allocating memory to objects, keeping track of which objects are still needed, and reclaiming memory when objects are no longer reachable.

When you write:

```python
name = "Gaurav"
```

Python creates or reuses a string object and binds the name `name` to that object. The name itself is not the object.

A simplified lifecycle:

1. **Allocation:** memory is obtained for an object.
2. **Initialization:** the object is given its initial state.
3. **Use:** names, containers and other objects hold references to it.
4. **Unreachable state:** no live part of the program can reach the object.
5. **Reclamation:** Python's memory manager can reclaim the object's memory.

Memory reclamation does not necessarily mean that the operating system immediately receives the memory back. Python may retain memory for reuse by later allocations.


## 2. Names, objects and references

In [ ]:
# A name refers to an object
a = [10, 20, 30]
b = a

print(a)
print(b)
print(a is b)       # Same object?
print(id(a), id(b)) # Same identity during this object's lifetime


### Rebinding versus modifying

- `b = a` creates another reference to the same list.
- `b = [10, 20, 30]` binds `b` to a different list.
- `a.append(40)` modifies the shared list object.
- `a = None` only rebinds `a`; it does not directly destroy the list if `b` still refers to it.


In [ ]:
first = [1, 2, 3]
second = first

second.append(4)
print("first:", first)   # The same list was modified

first = None
print("second:", second) # The list is still reachable


## 3. Stack frames, heap objects and references

A useful conceptual model is:

- **Call stack:** tracks active function calls and their local execution state (frames).
- **Heap:** holds Python objects and dynamically managed data.
- **References:** connect names, containers and frames to objects.

This is a simplified model, not a strict promise about the physical layout of every Python implementation. In CPython, local variables in a frame are references to objects; the objects themselves are managed by the object allocator and underlying memory allocator.


In [ ]:
def create_record():
    record = {"id": 101, "scores": [80, 90]}
    return record

result = create_record()
print(result)


## 4. Object allocation

Python usually allocates objects automatically when expressions create them. You normally do not call `malloc()` or manually reserve memory.

Examples of operations that can allocate objects:

- Creating lists, dictionaries, tuples, sets and strings.
- Constructing class instances.
- Creating function objects, iterators and generators.
- Building intermediate results during calculations.

Some immutable objects may be reused or interned as an implementation optimization. Do not rely on object identity for equal immutable values.


In [ ]:
numbers = [1, 2, 3]             # list allocation
profile = {"name": "Asha"}       # dictionary allocation
message = "Python memory"        # string object
point = (10, 20)                 # tuple object

class Sensor:
    def __init__(self, sensor_id):
        self.sensor_id = sensor_id

sensor = Sensor("S-101")         # instance allocation

print(type(numbers), type(profile), type(sensor))


## 5. `__new__()` and `__init__()`

For a typical user-defined class:

- `__new__()` creates and returns a new instance.
- `__init__()` initializes the already-created instance.
- `__init__()` does not allocate the instance itself.

Most application classes only need `__init__()`. Override `__new__()` for specialized cases such as immutable subclasses or controlled instance creation.


In [ ]:
class Device:
    def __new__(cls, *args, **kwargs):
        print("1. __new__: creating instance")
        instance = super().__new__(cls)
        return instance

    def __init__(self, model):
        print("2. __init__: initializing instance")
        self.model = model

device = Device("Edge-AI-01")
print(device.model)


## 6. Reference counting in CPython

CPython primarily uses **reference counting**.

An object's reference count roughly tracks how many strong references point to it. When the count reaches zero, CPython can usually deallocate the object immediately.

Important:
- `sys.getrefcount(obj)` itself temporarily adds a reference because the object is passed as an argument.
- The exact count is an implementation detail and can vary by Python version and execution context.
- Reference counting alone cannot reclaim unreachable reference cycles.


In [ ]:
import sys

data = []
print("Observed reference count:", sys.getrefcount(data))

alias = data
print("After another reference:", sys.getrefcount(data))

del alias
print("After deleting alias:", sys.getrefcount(data))


## 7. `del` does not mean “destroy this object”

`del name` removes a binding (or an item/attribute). It does not guarantee that the underlying object is destroyed.

The object can be reclaimed only when it is no longer reachable and the implementation's memory-management rules allow reclamation.


In [ ]:
items = ["A", "B", "C"]
backup = items

del items
print(backup)  # Still valid: backup references the list

del backup
# No remaining user-defined reference in this example.


## 8. Deallocation and object finalization

When an object is reclaimed, CPython releases the object's storage. For classes, `__del__()` may be called during finalization, but it is not a reliable resource-management mechanism.

Avoid relying on:
- The exact time `__del__()` runs.
- The order in which objects are finalized.
- Finalizers to close files, sockets or database connections.

Use context managers (`with`) for external resources. They provide deterministic cleanup when the block exits.


In [ ]:
class Demo:
    def __del__(self):
        print("Finalizer called (timing is not a general guarantee)")

obj = Demo()
del obj


## 9. Deterministic resource cleanup with context managers

Memory management and resource management are related but different.

A file handle, socket, lock or database connection is an external resource. Close it explicitly or use a context manager.


In [ ]:
from pathlib import Path

path = Path("memory_demo.txt")

with path.open("w", encoding="utf-8") as file:
    file.write("The file is closed when the with-block exits.")

print("File operation completed.")


## 10. Garbage collection: why reference counting is not enough

Consider two objects that refer to each other:

- Object A refers to object B.
- Object B refers to object A.
- No outside code can reach either object.

Their reference counts may remain above zero even though the pair is unreachable. This is a **reference cycle**.

CPython includes a cyclic garbage collector that can detect and reclaim many unreachable cycles.


In [ ]:
import gc

class Node:
    def __init__(self, name):
        self.name = name
        self.other = None

left = Node("left")
right = Node("right")
left.other = right
right.other = left

del left
del right

# The cycle may be collected by the cyclic garbage collector.
collected = gc.collect()
print("Objects collected:", collected)


## 11. Generational garbage collection

CPython's cyclic collector uses generations to reduce the cost of repeatedly examining every tracked object.

The broad idea:
- Newly created tracked objects are placed in the youngest generation.
- Objects that survive collections can be promoted.
- Younger objects are generally collected more frequently because many objects have short lifetimes.

Generation details and thresholds have changed across Python versions. Inspect the current runtime rather than memorizing fixed values.


In [ ]:
import gc

print("GC enabled:", gc.isenabled())
print("Collection thresholds:", gc.get_threshold())
print("Collection counts:", gc.get_count())
print("Tracked objects:", len(gc.get_objects()))


## 12. Inspecting and controlling the `gc` module

Useful functions:
- `gc.collect()` requests a collection and returns the number of collected objects.
- `gc.disable()` disables automatic cyclic collection.
- `gc.enable()` re-enables it.
- `gc.get_stats()` provides collection statistics.
- `gc.get_threshold()` and `gc.set_threshold()` inspect or adjust thresholds.

Do not disable garbage collection as a routine optimization. Measure first, and always restore the prior state if experimenting.


In [ ]:
import gc

print("Stats:", gc.get_stats())

collected = gc.collect()
print("Explicit collection:", collected)

# Example of safe state restoration when experimenting:
was_enabled = gc.isenabled()
try:
    gc.disable()
    # Run a short, controlled experiment here.
finally:
    if was_enabled:
        gc.enable()


## 13. Finding reference cycles

The `gc` module can help inspect objects tracked by the cyclic collector. Debugging options can expose unreachable objects that are difficult to reclaim, but debug modes may retain objects and affect memory usage.

Use these tools carefully in development, not indiscriminately in production.


In [ ]:
import gc

gc.set_debug(gc.DEBUG_SAVEALL)
try:
    gc.collect()
    print("Objects retained for inspection:", len(gc.garbage))
finally:
    gc.set_debug(0)
    gc.garbage.clear()


## 14. Shallow copy versus deep copy

Copying affects memory usage and object relationships.

- **Assignment:** creates another reference; no copy of the object.
- **Shallow copy:** creates a new outer container but reuses references to nested objects.
- **Deep copy:** recursively copies nested objects where supported.

Deep copying large structures can be expensive in both time and memory.


In [ ]:
import copy

original = {"user": "Asha", "scores": [70, 80]}

alias = original
shallow = copy.copy(original)
deep = copy.deepcopy(original)

original["scores"].append(90)

print("alias:", alias)
print("shallow:", shallow)
print("deep:", deep)


## 15. Mutable default arguments can retain data

Default argument expressions are evaluated once when the function is defined, not on every call. A mutable default can therefore retain state between calls.

Use `None` as the default and create a fresh object inside the function.


In [ ]:
# Avoid this pattern:
def add_item_bad(item, bucket=[]):
    bucket.append(item)
    return bucket

# Safer pattern:
def add_item(item, bucket=None):
    if bucket is None:
        bucket = []
    bucket.append(item)
    return bucket

print(add_item("A"))
print(add_item("B"))


## 16. Common sources of unintended memory retention

An object may remain alive because something still references it. Common causes include:

1. A global list or dictionary that grows without bounds.
2. Caches with no size or expiry policy.
3. Callbacks, closures or event handlers retaining large objects.
4. Long-lived containers holding completed task results.
5. Unclosed resources and objects that retain buffers.
6. Keeping unnecessary copies of large datasets.
7. Accumulating logs or results in memory instead of streaming them.

A memory leak in Python often means unintended retention, not necessarily a failure of the allocator to free memory.


In [ ]:
# A simple example of unbounded retention:
event_history = []

def record_event(event):
    event_history.append(event)

for i in range(5):
    record_event({"event_id": i, "payload": "sample" * 100})

print("Retained events:", len(event_history))

# Production systems should define a retention policy:
from collections import deque
recent_events = deque(maxlen=1000)


## 17. Weak references

A weak reference points to an object without keeping that object alive.

Useful applications include:
- Caches where entries should not extend object lifetime.
- Observer registries.
- Metadata associated with objects.

Not every object type supports weak references. A weak reference may return `None` after the object has been reclaimed.


In [ ]:
import weakref
import gc

class Model:
    pass

model = Model()
reference = weakref.ref(model)

print("Before deletion:", reference() is not None)

del model
gc.collect()

print("After deletion:", reference())


## 18. `__slots__` and per-instance memory

By default, many Python instances store attributes in an instance dictionary (`__dict__`). `__slots__` can reduce per-instance overhead for classes with a fixed set of attributes.

Trade-offs:
- Attribute names are restricted unless additional slots are provided.
- Weak references require a `__weakref__` slot when needed.
- Inheritance and multiple inheritance require care.
- Measure real workloads; slots are not automatically the best choice.


In [ ]:
class RegularPoint:
    def __init__(self, x, y):
        self.x = x
        self.y = y

class SlottedPoint:
    __slots__ = ("x", "y")

    def __init__(self, x, y):
        self.x = x
        self.y = y

regular = RegularPoint(2, 3)
slotted = SlottedPoint(2, 3)

print("Regular has __dict__:", hasattr(regular, "__dict__"))
print("Slotted has __dict__:", hasattr(slotted, "__dict__"))


## 19. Measuring object size with `sys.getsizeof()`

`sys.getsizeof(obj)` reports the shallow size of an object in bytes. It does not recursively include all objects referenced by that object.

For example, a list's reported size includes the list container's storage, not the full size of every element and nested object.


In [ ]:
import sys

values = [1, 2, 3, 4, 5]
nested = [[1, 2], [3, 4]]

print("List shallow size:", sys.getsizeof(values))
print("Nested list shallow size:", sys.getsizeof(nested))
print("One inner list:", sys.getsizeof(nested[0]))


## 20. Profiling allocations with `tracemalloc`

`tracemalloc` tracks Python memory allocations and can compare snapshots to identify where memory usage increased.

Workflow:
1. Start tracing.
2. Take a baseline snapshot.
3. Run the operation being investigated.
4. Take another snapshot.
5. Compare the snapshots.

It adds overhead, so use it for investigation and profiling rather than assuming traced memory equals total process memory.


In [ ]:
import tracemalloc

tracemalloc.start()

before = tracemalloc.take_snapshot()

records = [{"id": i, "values": list(range(20))} for i in range(2000)]

after = tracemalloc.take_snapshot()
for stat in after.compare_to(before, "lineno")[:5]:
    print(stat)

current, peak = tracemalloc.get_traced_memory()
print("Current traced bytes:", current)
print("Peak traced bytes:", peak)

tracemalloc.stop()


## 21. Streaming versus loading everything into memory

A memory-efficient pipeline often processes one record at a time instead of building a large intermediate list.

Compare:
- `readlines()` loads all lines into a list.
- Iterating over the file streams lines incrementally.
- A generator can transform records lazily.

Streaming is especially useful for large CSV files, logs, datasets and ETL pipelines.


In [ ]:
from pathlib import Path

# Create a small sample file.
sample_path = Path("large_log_sample.txt")
sample_path.write_text("\n".join(f"event-{i}" for i in range(1000)), encoding="utf-8")

# Loads all lines:
with sample_path.open(encoding="utf-8") as file:
    all_lines = file.readlines()

# Processes one line at a time:
def count_events(path):
    count = 0
    with path.open(encoding="utf-8") as file:
        for line in file:
            if line.startswith("event-"):
                count += 1
    return count

print("Total events:", count_events(sample_path))


## 22. Generators reduce intermediate memory

A list comprehension constructs a complete list. A generator expression produces values lazily.

Choose based on what the program needs:
- Use a list when you need indexing, repeated iteration or a materialized result.
- Use a generator when values can be consumed once, incrementally.


In [ ]:
import sys

squares_list = [number * number for number in range(10000)]
squares_generator = (number * number for number in range(10000))

print("List shallow size:", sys.getsizeof(squares_list))
print("Generator shallow size:", sys.getsizeof(squares_generator))

print("First generated values:", [next(squares_generator) for _ in range(5)])


## 23. Memory considerations in NumPy and machine learning

In data and ML workloads, memory is often dominated by arrays, tensors, copies and intermediate transformations.

Practical guidelines:
- Inspect `array.nbytes` for NumPy element storage.
- Choose an appropriate dtype (`float32` may use less memory than `float64`).
- Avoid unnecessary copies; understand views versus copies.
- Process large datasets in batches or chunks.
- Release references to large intermediates when they are no longer needed.
- Use generators or data loaders for streaming.
- Profile peak memory during training and inference.

Do not assume that deleting a Python variable instantly reduces the process's resident memory. Allocators and numerical libraries may retain memory for reuse.


In [ ]:
# Optional: run this cell if NumPy is installed.
try:
    import numpy as np

    values64 = np.ones((1000, 1000), dtype=np.float64)
    values32 = np.ones((1000, 1000), dtype=np.float32)

    print("float64 array bytes:", values64.nbytes)
    print("float32 array bytes:", values32.nbytes)

    # A view usually shares underlying data; a copy allocates new data.
    view = values32[:100, :]
    copied = values32[:100, :].copy()

    print("Shares memory with view:", np.shares_memory(values32, view))
    print("Shares memory with copy:", np.shares_memory(values32, copied))
except ImportError:
    print("NumPy is not installed. Install it with: pip install numpy")


## 24. `del`, garbage collection and process memory are different

These concepts are often confused:

- `del x`: removes a name binding or an item/attribute.
- Reference count reaches zero: CPython can reclaim that object.
- `gc.collect()`: asks the cyclic collector to look for unreachable cycles.
- Allocator frees a block: the Python allocator can reuse it.
- Operating system reports lower RSS: process resident memory has decreased.

These events are related, but they are not identical and may not happen at the same time.


## 25. A practical memory investigation workflow

When an application consumes more memory than expected:

1. Reproduce the workload with realistic input sizes.
2. Record baseline and peak process memory.
3. Use `tracemalloc` snapshots to locate Python allocation growth.
4. Check long-lived lists, dictionaries, caches and closures.
5. Look for reference cycles if objects are unexpectedly retained.
6. Inspect copies and intermediate arrays in data pipelines.
7. Change one thing at a time and benchmark again.
8. Verify correctness and performance after optimization.

Use the right metric: Python-traced allocations, object sizes, array bytes and process RSS measure different things.


## 26. Quick comparison: core memory-management concepts

| Concept | Meaning |
|---|---|
| Allocation | Obtaining memory for a new object |
| Binding | Associating a name with an object |
| Reference | A link that keeps an object reachable |
| Rebinding | Making a name refer to another object |
| `del` | Removing a binding/item/attribute |
| Reference counting | Tracking strong references in CPython |
| Cyclic GC | Detecting unreachable reference cycles |
| Deallocation | Reclaiming an object's storage |
| Weak reference | Reference that does not keep target alive |
| `tracemalloc` | Tracking Python memory allocations |
| Context manager | Deterministic cleanup of managed resources |


# Advanced practice problems

Solve these without looking for ready-made solutions. Design, implement, test and profile your own approach. Each problem is intended to combine multiple concepts.

## Problem 1 — Memory-Aware Log Processing Engine

Build a log-processing engine that can process multi-gigabyte log files without loading the entire file into memory.

**Requirements**
- Implement a generator that reads records incrementally.
- Filter records by severity and date.
- Transform each accepted record into a structured object.
- Maintain only a configurable rolling window of recent errors.
- Add a cache for repeated parsing, with a maximum size.
- Provide a memory profiling mode using `tracemalloc`.

**Design considerations**
- Compare list-based processing with generator-based processing.
- Prevent unbounded cache growth.
- Avoid retaining raw records after transformation.
- Include malformed-line handling and summary statistics.

**Acceptance tests**
- Process a generated file with at least one million records.
- Confirm the rolling window never exceeds its configured size.
- Compare peak traced memory for streaming and list-based approaches.
- Verify malformed records do not stop the pipeline.


## Problem 2 — Object Lifecycle and Reference-Cycle Inspector

Create a small diagnostic tool that demonstrates and investigates object lifecycles.

**Requirements**
- Define linked objects that can reference one another.
- Build both acyclic structures and cyclic graphs.
- Use weak references to observe object lifetime.
- Use `gc` to inspect collection behavior.
- Provide a report containing object labels, weak-reference status and collection results.
- Demonstrate why `del` on one variable may not reclaim an object.

**Design considerations**
- Do not depend on exact reference counts.
- Avoid treating `__del__()` output as proof of reliable cleanup.
- Keep the experiment repeatable and isolate each test case.
- Document CPython-specific observations separately from Python guarantees.

**Acceptance tests**
- Show an object remaining alive while a strong reference exists.
- Show a weak reference becoming dead after all strong references are removed and collection occurs.
- Demonstrate an unreachable cycle being collected.
- Explain the observed results in a short report.


## Problem 3 — Bounded Cache with Weak References and Expiry

Design a reusable cache for an application that processes expensive objects.

**Requirements**
- Support a maximum number of entries.
- Support time-based expiry.
- Provide a policy for least-recently-used eviction.
- Support weakly held values where the value type permits it.
- Expose hit, miss, expiry and eviction counters.
- Ensure cache metadata does not accidentally retain values.

**Design considerations**
- Decide which values should be strongly held and which may be weakly held.
- Handle objects that do not support weak references.
- Define behavior when a weakly referenced value disappears.
- Keep cache operations safe if used by multiple threads.

**Acceptance tests**
- Verify the cache never exceeds its configured capacity.
- Verify expired entries are not returned.
- Verify a weakly held object can be reclaimed when no other strong reference exists.
- Test concurrent reads and writes.


## Problem 4 — Memory-Efficient ML Dataset Pipeline

Build a data preparation pipeline for a large tabular machine-learning dataset.

**Requirements**
- Read data in chunks.
- Validate and transform each chunk.
- Use suitable numeric dtypes.
- Avoid unnecessary intermediate copies.
- Support batch feature generation.
- Provide a generator that yields training batches.
- Track peak memory and processing time.

**Design considerations**
- Compare `float64` and `float32` where accuracy permits.
- Identify operations that create copies.
- Release chunk-specific references after processing.
- Separate data loading, validation, transformation and batching into components.

**Acceptance tests**
- Process a synthetic dataset larger than the available comfortable in-memory working set by using chunks.
- Report memory usage per chunk and peak traced memory.
- Verify batch sizes and output schema.
- Compare the result with a smaller fully in-memory reference dataset.


## Problem 5 — Long-Running Task Worker Memory Audit

Develop a worker that handles thousands of jobs over a long-running process lifetime.

**Requirements**
- Implement a job queue and worker loop.
- Process jobs using a configurable handler registry.
- Record job metrics without retaining full job payloads indefinitely.
- Add bounded result history and bounded caches.
- Use context managers for temporary files.
- Capture periodic `tracemalloc` snapshots.
- Provide a diagnostic report that identifies allocation growth by source line.

**Design considerations**
- Look for references retained by callbacks, closures and global registries.
- Ensure exceptions do not leave jobs or resources in an inconsistent state.
- Distinguish normal allocator reuse from unintended object retention.
- Make shutdown graceful and testable.

**Acceptance tests**
- Run at least 10,000 generated jobs.
- Confirm result history and cache sizes remain bounded.
- Compare early-run and late-run memory snapshots.
- Verify temporary resources are closed after success and failure.
- Document any remaining memory growth and its cause.


## Final revision checklist

- [ ] I can explain the difference between a name and an object.
- [ ] I understand allocation, initialization, references and deallocation.
- [ ] I can explain CPython reference counting.
- [ ] I understand why reference cycles need cyclic garbage collection.
- [ ] I can use `gc` for basic inspection.
- [ ] I know why `del` does not guarantee immediate memory release.
- [ ] I can distinguish shallow copy and deep copy.
- [ ] I can use weak references appropriately.
- [ ] I understand the limits of `sys.getsizeof()`.
- [ ] I can profile allocation growth with `tracemalloc`.
- [ ] I can use generators and chunking to reduce peak memory.
- [ ] I can identify common causes of unintended object retention.
- [ ] I can explain why process RSS may not fall after objects are reclaimed.
